# NBA Feature Engineering

This notebook implements feature engineering for NBA predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from datetime import datetime, timedelta
from src.utils.database import get_session, Team, Game, TeamStats, Player, PlayerStats
from src.features.structured_features import StructuredFeatures, get_player_features
from src.features.feature_utils import prepare_game_features

## Team Features

In [ ]:
session = get_session()
structured = StructuredFeatures(session)

teams = session.query(Team).limit(5).all()
as_of_date = datetime.now()

for team in teams:
    features = structured.get_team_features(team.id, as_of_date)
    print(f"{team.name}: {len(features)} features generated")

## Rest Features

In [ ]:
for team in teams[:3]:
    game_date = as_of_date + timedelta(days=1)
    rest_features = structured.get_rest_features(team.id, game_date)
    print(f"{team.name}: {rest_features}")

## Strength of Schedule

In [ ]:
for team in teams[:3]:
    sos_features = structured.get_sos_features(team.id, as_of_date)
    print(f"{team.name} SOS: {sos_features}")

## Player Features

In [ ]:
players = session.query(Player).limit(5).all()

for player in players:
    features = get_player_features(player.id, as_of_date, session)
    print(f"{player.name}: {len(features)} features")

## Game Features (Early Fusion)

In [ ]:
if len(teams) >= 2:
    home_team = teams[0]
    away_team = teams[1]
    
    game_features = prepare_game_features(home_team.id, away_team.id, as_of_date, session)
    print(f"Total game features: {len(game_features)}")
    print(f"Feature names: {list(game_features.keys())[:10]}...")

## Feature Correlation Analysis

In [ ]:
team_stats = pd.read_sql(session.query(TeamStats).statement, session.bind)

numeric_cols = ['offensive_rating', 'defensive_rating', 'net_rating', 'pace', 'effective_fg_pct']
corr_matrix = team_stats[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Summary

- Generated team-level features with rolling windows (5, 10, 20 games)
- Created rest and travel features
- Implemented strength of schedule calculation
- Generated player-level features for props predictions
- Built early fusion game features combining structured stats with future embedding placeholders